# T049 - T036 XGBoost + label-free entity features\n\nT036 depth12/leaves511 on T036 features plus 16 label-free item/creator features over train+test (upload age, backward exposure counts, entity mean within-request rank of feature54/409/506). Original holdout; compare with T036 0.88967/0.88959. GPU, no submission.

In [ ]:
import json
import os
from pathlib import Path
import subprocess
import sys

sources = {'feature_screen.py': '"""Local LightGBM screen of request-context, train+test count and target-encoding features.\n\nTrains on rows before the original primary cutoff and scores the original\n307,528-row holdout. request_id is used only for grouping, never as an input.\n"""\nimport argparse\nfrom datetime import datetime\nimport gc\nimport json\nfrom pathlib import Path\nimport time\n\nimport lightgbm as lgb\nimport numpy as np\nimport polars as pl\nfrom sklearn.metrics import roc_auc_score\nfrom sklearn.model_selection import GroupKFold\n\nCUTOFF = datetime.fromisoformat("2026-10-28T08:06:52.397000")\nUSER, ITEM = "feature5", "feature18"\nLOW_CATEGORICAL = [f"feature{i}" for i in [2, 4, 6, 7, 8, 9, 10, 11, 12, 13, 14, 16, 17, 19, 21, 22, 25]]\nCOUNT_KEYS = [USER, ITEM, "feature23", "feature20", "feature19", "feature9", "feature12"]\nENCODED = [ITEM, "feature23", "feature20", "feature19", "feature9", "feature12", "feature21", USER]\nRANKED = ["feature418", "feature419", "feature411", "feature410", "feature430",\n          "feature431", "feature417", "feature352", "feature346", "feature480"]\nSMOOTHING = 20.0\n\n\ndef scalar_numeric(schema):\n    skip = {"feature1", "feature3"}\n    return [c for c, t in schema.items()\n            if c.startswith("feature") and c not in skip\n            and str(t) in {"Float32", "Int64", "Int8"}]\n\n\ndef time_seconds(column):\n    return pl.col(column).dt.epoch("ms").cast(pl.Float64) / 1000.0\n\n\ndef count_features(keys):\n    """Label-free counts over the complete train+test population."""\n    frame = keys.with_columns(time_seconds("feature1").alias("t"))\n    exprs = []\n    for key in COUNT_KEYS:\n        exprs += [pl.len().over(key).alias(f"cnt_{key}"),\n                  pl.col("request_id").n_unique().over(key).alias(f"req_{key}")]\n    frame = frame.with_columns(exprs)\n    pair = [USER, ITEM]\n    frame = frame.with_columns(\n        pl.len().over(pair).alias("cnt_user_item"),\n        pl.col("t").min().over(ITEM).alias("item_first"),\n        pl.col("t").max().over(ITEM).alias("item_last"),\n        pl.col("t").min().over(USER).alias("user_first"),\n        pl.col("t").max().over(USER).alias("user_last"),\n        pl.len().over([ITEM, pl.col("feature1").dt.truncate("1h")]).alias("cnt_item_hour"),\n        pl.len().over([ITEM, pl.col("feature1").dt.truncate("1d")]).alias("cnt_item_day"),\n    )\n    requests = (\n        frame.group_by("request_id").agg(pl.col(USER).first(), pl.col("t").first())\n        .sort([USER, "t", "request_id"])\n        .with_columns(\n            pl.int_range(pl.len()).over(USER).alias("user_req_index"),\n            (pl.col("t") - pl.col("t").shift(1).over(USER)).alias("user_prev_gap"),\n            (pl.col("t").shift(-1).over(USER) - pl.col("t")).alias("user_next_gap"),\n        )\n        .select("request_id", "user_req_index", "user_prev_gap", "user_next_gap")\n    )\n    frame = frame.join(requests, on="request_id", how="left")\n    pairs = (\n        frame.select(USER, ITEM, "t").unique().sort([USER, ITEM, "t"])\n        .with_columns(\n            pl.int_range(pl.len()).over([USER, ITEM]).alias("user_item_index"),\n            (pl.col("t") - pl.col("t").shift(1).over([USER, ITEM])).alias("user_item_prev_gap"),\n            (pl.col("t").shift(-1).over([USER, ITEM]) - pl.col("t")).alias("user_item_next_gap"),\n        )\n    )\n    frame = frame.join(pairs, on=[USER, ITEM, "t"], how="left")\n    frame = frame.with_columns(\n        (pl.col("req_feature5") - pl.col("user_req_index") - 1).alias("user_reqs_after"),\n        (pl.col("t") - pl.col("item_first")).alias("item_age"),\n        (pl.col("item_last") - pl.col("t")).alias("item_remaining"),\n        (pl.col("t") - pl.col("user_first")).alias("user_since_first"),\n        (pl.col("user_last") - pl.col("t")).alias("user_until_last"),\n    ).drop("item_first", "item_last", "user_first", "user_last")\n    names = [c for c in frame.columns if c not in keys.columns and c != "t"]\n    return frame.select("row", *names), names\n\n\ndef request_features(frame):\n    """Within-request context; frame is one split population with numeric columns."""\n    group = "request_id"\n    exprs = [pl.len().over(group).alias("rq_size"),\n             pl.col(ITEM).n_unique().over(group).alias("rq_items"),\n             pl.len().over([group, ITEM]).alias("rq_item_dup")]\n    for c in RANKED:\n        exprs += [\n            (pl.col(c).rank("average").over(group) / pl.len().over(group)).alias(f"rq_rank_{c}"),\n            (pl.col(c) - pl.col(c).mean().over(group)).alias(f"rq_dev_{c}"),\n            (pl.col(c) - pl.col(c).max().over(group)).alias(f"rq_gap_{c}"),\n        ]\n    out = frame.select("row", *exprs)\n    return out, [c for c in out.columns if c != "row"]\n\n\ndef target_encode(train, valid, column, folds):\n    prior = float(train["watched"].mean())\n    y = train["watched"].to_numpy().astype(np.float64)\n    keys = train[column]\n    oof = np.empty(len(train))\n    support = np.empty(len(train))\n    for fit, apply in folds:\n        stats = (pl.DataFrame({"k": keys[fit], "y": y[fit]})\n                 .group_by("k").agg(pl.col("y").sum().alias("s"), pl.len().alias("n")))\n        joined = pl.DataFrame({"k": keys[apply]}).join(stats, on="k", how="left").fill_null(0)\n        s, n = joined["s"].to_numpy(), joined["n"].to_numpy()\n        oof[apply] = (s + SMOOTHING * prior) / (n + SMOOTHING)\n        support[apply] = n\n    stats = (pl.DataFrame({"k": keys, "y": y})\n             .group_by("k").agg(pl.col("y").sum().alias("s"), pl.len().alias("n")))\n    joined = pl.DataFrame({"k": valid[column]}).join(stats, on="k", how="left").fill_null(0)\n    s, n = joined["s"].to_numpy(), joined["n"].to_numpy()\n    held = (s + SMOOTHING * prior) / (n + SMOOTHING)\n    return (oof, support), (held, n)\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--data", type=Path, default=Path("data"))\n    parser.add_argument("--output", type=Path, default=Path("artifacts/local-t023"))\n    parser.add_argument("--variants", default="base,request,counts,target,all")\n    parser.add_argument("--learning-rate", type=float, default=0.05)\n    parser.add_argument("--rounds", type=int, default=4000)\n    parser.add_argument("--threads", type=int, default=8)\n    parser.add_argument("--rows", type=int, default=0, help="Keep 1/N of requests for smoke tests")\n    args = parser.parse_args()\n    args.output.mkdir(parents=True, exist_ok=True)\n    start = time.time()\n    schema = pl.scan_parquet(args.data / "train.parquet").collect_schema()\n    numeric = scalar_numeric(schema)\n    key_columns = ["request_id", "feature1", *sorted(set(COUNT_KEYS + ENCODED + LOW_CATEGORICAL), key=lambda c: int(c[7:]))]\n    train = pl.scan_parquet(args.data / "train.parquet").with_row_index("row")\n    if args.rows:\n        train = train.filter(pl.col("request_id").hash(7) % args.rows == 0)\n    train = train.select("row", "watched", "feature3", *key_columns, *numeric).collect()\n    test_keys = pl.scan_parquet(args.data / "test.parquet").select(*key_columns)\n    if args.rows:\n        test_keys = test_keys.filter(pl.col("request_id").hash(7) % args.rows == 0)\n    test_keys = test_keys.collect()\n    offset = int(train["row"].max()) + 1\n    keys = pl.concat([\n        train.select("row", *key_columns),\n        test_keys.with_row_index("row", offset=offset).with_columns(pl.col("row").cast(train["row"].dtype)),\n    ])\n    del test_keys\n    counts, count_names = count_features(keys)\n    del keys\n    gc.collect()\n    train = train.join(counts, on="row", how="left")\n    del counts\n    rq, rq_names = request_features(train.select("row", "request_id", ITEM, *RANKED))\n    train = train.join(rq, on="row", how="left")\n    del rq\n    is_valid = (train["feature1"] >= CUTOFF).to_numpy()\n    fit_rows, valid_rows = train.filter(~pl.Series(is_valid)), train.filter(pl.Series(is_valid))\n    del train\n    gc.collect()\n    folds = list(GroupKFold(5).split(np.zeros(len(fit_rows)), groups=fit_rows["request_id"].to_numpy()))\n    te_fit, te_valid, te_names = {}, {}, []\n    for column in ENCODED:\n        (a, an), (b, bn) = target_encode(fit_rows, valid_rows, column, folds)\n        te_fit[f"te_{column}"], te_valid[f"te_{column}"] = a, b\n        te_fit[f"ten_{column}"], te_valid[f"ten_{column}"] = an, bn\n        te_names += [f"te_{column}", f"ten_{column}"]\n    categories = {}\n    for c in LOW_CATEGORICAL:\n        vocabulary = fit_rows[c].drop_nulls().unique().sort()\n        categories[c] = vocabulary\n    base = ["feature3", *numeric, *LOW_CATEGORICAL]\n    groups = {"base": base, "request": base + rq_names, "counts": base + count_names,\n              "target": base + te_names, "all": base + rq_names + count_names + te_names}\n\n    def matrix(frame, te, columns):\n        parts = []\n        for c in columns:\n            if c in te:\n                parts.append(te[c].astype(np.float32))\n            elif c in categories:\n                codes = frame.select(pl.col(c).cast(pl.Enum(categories[c].to_list()), strict=False)\n                                     .to_physical().cast(pl.Float32))[c].to_numpy()\n                parts.append(codes)\n            else:\n                parts.append(frame[c].cast(pl.Float32).to_numpy())\n        return np.ascontiguousarray(np.column_stack(parts), dtype=np.float32)\n\n    y_fit = fit_rows["watched"].to_numpy()\n    y_valid = valid_rows["watched"].to_numpy()\n    valid_ids = valid_rows["row"].to_numpy()\n    results = {}\n    for name in args.variants.split(","):\n        columns = groups[name]\n        began = time.time()\n        x_fit = matrix(fit_rows, te_fit, columns)\n        x_valid = matrix(valid_rows, te_valid, columns)\n        categorical = [i for i, c in enumerate(columns) if c in categories]\n        train_set = lgb.Dataset(x_fit, y_fit, feature_name=columns,\n                                categorical_feature=categorical, free_raw_data=True)\n        valid_set = lgb.Dataset(x_valid, y_valid, reference=train_set)\n        params = {\n            "objective": "binary", "metric": "auc", "learning_rate": args.learning_rate,\n            "num_leaves": 127, "min_data_in_leaf": 200, "feature_fraction": 0.5,\n            "bagging_fraction": 0.8, "bagging_freq": 1, "lambda_l2": 10,\n            "max_bin": 127, "seed": 42, "num_threads": args.threads, "verbosity": -1,\n            "cat_smooth": 20, "max_cat_to_onehot": 8,\n        }\n        log = {}\n        model = lgb.train(params, train_set, args.rounds, valid_sets=[valid_set], valid_names=["holdout"],\n                          callbacks=[lgb.early_stopping(200, verbose=False), lgb.record_evaluation(log),\n                                     lgb.log_evaluation(250)])\n        prediction = model.predict(x_valid, num_iteration=model.best_iteration)\n        auc = float(roc_auc_score(y_valid, prediction))\n        curve = log["holdout"]["auc"]\n        importance = sorted(zip(columns, model.feature_importance("gain")), key=lambda p: -p[1])[:30]\n        results[name] = {\n            "holdout_auc": auc, "best_iteration": model.best_iteration,\n            "auc_at_140": curve[min(139, len(curve) - 1)], "features": len(columns),\n            "seconds": time.time() - began,\n            "top_gain": [[c, float(g)] for c, g in importance],\n        }\n        np.save(args.output / f"valid-{name}.npy", prediction.astype(np.float32))\n        print(name, json.dumps({k: v for k, v in results[name].items() if k != "top_gain"}), flush=True)\n        print(" top:", [c for c, _ in importance[:15]], flush=True)\n        (args.output / "results.json").write_text(json.dumps({\n            "cutoff": CUTOFF.isoformat(), "valid_rows": int(len(y_valid)),\n            "fit_rows": int(len(y_fit)), "early_stopping_on_holdout": True,\n            "results": results, "seconds": time.time() - start}, indent=2) + "\\n")\n        del x_fit, x_valid, train_set, valid_set, model\n        gc.collect()\n    np.save(args.output / "valid-row-ids.npy", valid_ids)\n\n\nif __name__ == "__main__":\n    main()\n', 'request_screen.py': '"""T026 local LightGBM screen: wider within-request context features.\n\nSame protocol as T023 (train before the original cutoff, score the original\n307,528-row holdout, early stopping on the holdout for every variant, so deltas\nare comparable). Wide features are chosen by gain from a fixed-budget base\nmodel fitted on training rows only; the holdout is never used to choose them.\n"""\nimport argparse\nimport gc\nimport json\nfrom pathlib import Path\nimport time\n\nimport lightgbm as lgb\nimport numpy as np\nimport polars as pl\nfrom sklearn.metrics import roc_auc_score\n\nfrom feature_screen import CUTOFF, ITEM, LOW_CATEGORICAL, RANKED, scalar_numeric\n\nPARAMS = {\n    "objective": "binary", "metric": "auc", "num_leaves": 127, "min_data_in_leaf": 200,\n    "feature_fraction": 0.5, "bagging_fraction": 0.8, "bagging_freq": 1, "lambda_l2": 10,\n    "max_bin": 127, "seed": 42, "verbosity": -1, "cat_smooth": 20, "max_cat_to_onehot": 8,\n}\n\n\ndef context(columns, stats=()):\n    group = "request_id"\n    size = pl.len().over(group)\n    exprs = [size.alias("rq_size"), pl.col(ITEM).n_unique().over(group).alias("rq_items"),\n             pl.len().over([group, ITEM]).alias("rq_item_dup")]\n    for c in columns:\n        exprs += [(pl.col(c).rank("average").over(group) / size).alias(f"rq_rank_{c}"),\n                  (pl.col(c) - pl.col(c).mean().over(group)).alias(f"rq_dev_{c}"),\n                  (pl.col(c) - pl.col(c).max().over(group)).alias(f"rq_gap_{c}")]\n    for c in stats:\n        exprs += [pl.col(c).mean().over(group).alias(f"rq_mean_{c}"),\n                  pl.col(c).std().over(group).alias(f"rq_std_{c}")]\n    return [e.cast(pl.Float32) for e in exprs]\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--data", type=Path, default=Path("data"))\n    parser.add_argument("--output", type=Path, default=Path("artifacts/local-t026"))\n    parser.add_argument("--variants", default="control,no_size,wide,wide_stats")\n    parser.add_argument("--wide", type=int, default=50)\n    parser.add_argument("--threads", type=int, default=6)\n    parser.add_argument("--rows", type=int, default=0, help="Keep 1/N of requests for smoke tests")\n    args = parser.parse_args()\n    args.output.mkdir(parents=True, exist_ok=True)\n    start = time.time()\n    numeric = scalar_numeric(pl.scan_parquet(args.data / "train.parquet").collect_schema())\n    frame = pl.scan_parquet(args.data / "train.parquet")\n    if args.rows:\n        frame = frame.filter(pl.col("request_id").hash(7) % args.rows == 0)\n    frame = frame.select("watched", "feature1", "feature3", "request_id", ITEM,\n                         *dict.fromkeys([*LOW_CATEGORICAL, *numeric])).collect()\n    is_valid = (frame["feature1"] >= CUTOFF).to_numpy()\n    categories = {c: frame.filter(pl.Series(~is_valid))[c].drop_nulls().unique().sort().to_list()\n                  for c in LOW_CATEGORICAL}\n    base = ["feature3", *numeric, *LOW_CATEGORICAL]\n\n    def matrix(source, columns):\n        parts = []\n        for c in columns:\n            column = pl.col(c)\n            if c in categories:\n                column = column.cast(pl.Enum(categories[c]), strict=False).to_physical()\n            parts.append(source.select(column.cast(pl.Float32))[c].to_numpy())\n        return np.ascontiguousarray(np.column_stack(parts), dtype=np.float32)\n\n    y = frame["watched"].to_numpy()\n    y_fit, y_valid = y[~is_valid], y[is_valid]\n    categorical = lambda columns: [i for i, c in enumerate(columns) if c in categories]\n    x = matrix(frame.filter(pl.Series(~is_valid)), base)\n    selector = lgb.train({**PARAMS, "learning_rate": .05, "num_threads": args.threads},\n                         lgb.Dataset(x, y_fit, feature_name=base, categorical_feature=categorical(base)), 150)\n    gains = dict(zip(base, selector.feature_importance("gain")))\n    del x, selector\n    gc.collect()\n    candidates = [c for c in numeric if c not in categories and c not in {"feature3"}]\n    wide = sorted(candidates, key=lambda c: -gains[c])[:args.wide]\n    wide = list(dict.fromkeys([*RANKED, *wide]))\n    print(f"wide context over {len(wide)} features; top: {wide[10:20]}", flush=True)\n    frame = frame.with_columns(context(wide, wide[:10]))\n    control = ["rq_size", "rq_items", "rq_item_dup"] + [f"rq_{k}_{c}" for c in RANKED for k in ["rank", "dev", "gap"]]\n    wide_names = ["rq_size", "rq_items", "rq_item_dup"] + [f"rq_{k}_{c}" for c in wide for k in ["rank", "dev", "gap"]]\n    groups = {\n        "control": base + control,\n        "no_size": base + [c for c in control if c != "rq_size"],\n        "wide": base + wide_names,\n        "wide_stats": base + wide_names + [f"rq_{k}_{c}" for c in wide[:10] for k in ["mean", "std"]],\n    }\n    fit_rows, valid_rows = frame.filter(pl.Series(~is_valid)), frame.filter(pl.Series(is_valid))\n    del frame\n    gc.collect()\n    results = {}\n    for name in args.variants.split(","):\n        columns, began = groups[name], time.time()\n        train_set = lgb.Dataset(matrix(fit_rows, columns), y_fit, feature_name=columns,\n                                categorical_feature=categorical(columns), free_raw_data=True)\n        x_valid = matrix(valid_rows, columns)\n        valid_set = lgb.Dataset(x_valid, y_valid, reference=train_set)\n        model = lgb.train({**PARAMS, "learning_rate": .05, "num_threads": args.threads}, train_set, 4000,\n                          valid_sets=[valid_set], valid_names=["holdout"],\n                          callbacks=[lgb.early_stopping(200, verbose=False), lgb.log_evaluation(250)])\n        auc = float(roc_auc_score(y_valid, model.predict(x_valid, num_iteration=model.best_iteration)))\n        top = sorted(zip(columns, model.feature_importance("gain")), key=lambda p: -p[1])[:30]\n        results[name] = {"holdout_auc": auc, "best_iteration": model.best_iteration,\n                         "features": len(columns), "seconds": time.time() - began,\n                         "top_gain": [[c, float(g)] for c, g in top]}\n        print(name, json.dumps({k: v for k, v in results[name].items() if k != "top_gain"}), flush=True)\n        (args.output / "results.json").write_text(json.dumps({\n            "cutoff": CUTOFF.isoformat(), "valid_rows": int(len(y_valid)), "fit_rows": int(len(y_fit)),\n            "early_stopping_on_holdout": True, "wide_features": wide, "results": results,\n            "seconds": time.time() - start}, indent=2) + "\\n")\n        del train_set, valid_set, x_valid, model\n        gc.collect()\n\n\nif __name__ == "__main__":\n    main()\n', 'classic_screen.py': '"""T029 local screen: classic learners on the T026 wide request-context features.\n\nSame protocol as T026 (fit rows before the original cutoff, score the original\n307,528-row holdout, early stopping on the holdout). The wide feature list is the\none T026 chose from training rows only. Holdout predictions are saved with file\nrow ids so they can be blended with the neural models.\n"""\nimport argparse\nimport gc\nimport json\nfrom pathlib import Path\nimport time\n\nimport numpy as np\nimport polars as pl\nfrom sklearn.metrics import roc_auc_score\n\nfrom feature_screen import CUTOFF, ITEM, LOW_CATEGORICAL, scalar_numeric\nfrom request_screen import context\n\n\ndef fit_xgboost(x_fit, y_fit, x_valid, y_valid, categorical, threads):\n    import xgboost as xgb\n\n    fit = xgb.QuantileDMatrix(x_fit, y_fit, max_bin=127)\n    valid = xgb.QuantileDMatrix(x_valid, y_valid, ref=fit, max_bin=127)\n    params = {\n        "objective": "binary:logistic", "eval_metric": "auc", "tree_method": "hist",\n        "max_depth": 0, "max_leaves": 127, "grow_policy": "lossguide", "min_child_weight": 20,\n        "eta": 0.05, "subsample": 0.8, "colsample_bytree": 0.5, "lambda": 10,\n        "nthread": threads, "seed": 42, "max_bin": 127,\n    }\n    model = xgb.train(params, fit, 4000, evals=[(valid, "holdout")],\n                      early_stopping_rounds=200, verbose_eval=250)\n    predictions = model.predict(valid, iteration_range=(0, model.best_iteration + 1))\n    return predictions, {"best_iteration": int(model.best_iteration)}\n\n\ndef fit_catboost(x_fit, y_fit, x_valid, y_valid, categorical, threads):\n    from catboost import CatBoostClassifier, Pool\n\n    def pool(x, y):\n        frame = {f"f{i}": x[:, i] for i in range(x.shape[1])}\n        for i in categorical:\n            frame[f"f{i}"] = np.nan_to_num(x[:, i], nan=-1).astype(np.int32)\n        import pandas as pd\n\n        return Pool(pd.DataFrame(frame), y, cat_features=[f"f{i}" for i in categorical])\n\n    fit, valid = pool(x_fit, y_fit), pool(x_valid, y_valid)\n    model = CatBoostClassifier(\n        iterations=4000, learning_rate=0.08, depth=8, border_count=128, l2_leaf_reg=10,\n        eval_metric="AUC", od_type="Iter", od_wait=200, thread_count=threads,\n        random_seed=42, verbose=250, use_best_model=True,\n    )\n    model.fit(fit, eval_set=valid)\n    return model.predict_proba(valid)[:, 1], {"best_iteration": int(model.get_best_iteration())}\n\n\ndef fit_logreg(x_fit, y_fit, x_valid, y_valid, categorical, threads):\n    from sklearn.linear_model import LogisticRegression\n    from sklearn.preprocessing import OneHotEncoder\n\n    sample = np.sort(np.random.default_rng(42).choice(len(y_fit), min(400_000, len(y_fit)), replace=False))\n    x_fit, y_fit = x_fit[sample], y_fit[sample]\n    dense = [i for i in range(x_fit.shape[1]) if i not in set(categorical)]\n\n    keep = np.isnan(x_fit[:, dense]).any(axis=0)\n\n    def transform(x, stats=None):\n        values = x[:, dense]\n        if stats is None:\n            low, high = np.nanpercentile(values, [0.5, 99.5], axis=0)\n            clipped = np.clip(values, low, high)\n            stats = (low, high, np.nanmean(clipped, axis=0), np.nanstd(clipped, axis=0) + 1e-6)\n        low, high, mean, std = stats\n        missing = np.isnan(values)\n        values = (np.clip(values, low, high) - mean) / std\n        values[missing] = 0\n        return np.hstack([values, missing[:, keep]]).astype(np.float32), stats\n\n    fit_values, stats = transform(x_fit)\n    valid_values, _ = transform(x_valid, stats)\n    encoder = OneHotEncoder(handle_unknown="ignore", min_frequency=50, sparse_output=False, dtype=np.float32)\n    fit_codes = encoder.fit_transform(np.nan_to_num(x_fit[:, categorical], nan=-1))\n    valid_codes = encoder.transform(np.nan_to_num(x_valid[:, categorical], nan=-1))\n    fit_values, valid_values = np.hstack([fit_values, fit_codes]), np.hstack([valid_values, valid_codes])\n    model = LogisticRegression(C=0.1, max_iter=500, tol=1e-5)\n    model.fit(fit_values, y_fit)\n    return model.predict_proba(valid_values)[:, 1], {"iterations": int(model.n_iter_[0]),\n                                                    "inputs": int(fit_values.shape[1])}\n\n\ndef build_matrices(data, wide, rows=0):\n    """Fit/holdout float32 matrices of the T029 wide feature set (categoricals as fit-period codes)."""\n    numeric = scalar_numeric(pl.scan_parquet(data / "train.parquet").collect_schema())\n    frame = pl.scan_parquet(data / "train.parquet").with_row_index("_row_id")\n    if rows:\n        frame = frame.filter(pl.col("request_id").hash(7) % rows == 0)\n    frame = frame.select("_row_id", "watched", "feature1", "feature3", "request_id", ITEM,\n                         *dict.fromkeys([*LOW_CATEGORICAL, *numeric])).collect()\n    frame = frame.with_columns(context(wide))\n    is_valid = (frame["feature1"] >= CUTOFF).to_numpy()\n    categories = {c: frame.filter(pl.Series(~is_valid))[c].drop_nulls().unique().sort().to_list()\n                  for c in LOW_CATEGORICAL}\n    columns = ["feature3", *numeric, *LOW_CATEGORICAL, "rq_size", "rq_items", "rq_item_dup",\n               *[f"rq_{k}_{c}" for c in wide for k in ["rank", "dev", "gap"]]]\n    categorical = [i for i, c in enumerate(columns) if c in categories]\n\n    y = frame["watched"].to_numpy()\n    y_fit, y_valid = y[~is_valid], y[is_valid]\n    row_ids = frame["_row_id"].to_numpy()[is_valid]\n    x_fit = np.empty((int((~is_valid).sum()), len(columns)), dtype=np.float32)\n    x_valid = np.empty((int(is_valid.sum()), len(columns)), dtype=np.float32)\n    for i, c in enumerate(columns):\n        column = pl.col(c)\n        if c in categories:\n            column = column.cast(pl.Enum(categories[c]), strict=False).to_physical()\n        values = frame.select(column.cast(pl.Float32))[c].to_numpy()\n        x_fit[:, i], x_valid[:, i] = values[~is_valid], values[is_valid]\n\n    del frame\n    gc.collect()\n    return x_fit, y_fit, x_valid, y_valid, row_ids, columns, categorical\n\n\ndef full_matrices(data, wide, rows=0, fit_only=False):\n    """All-train and test matrices of the T029 wide feature set for a full-data refit.\n\n    Categorical codes come from every labeled row (only pre-cutoff rows with `fit_only`);\n    request context is computed per request within each file (requests never cross\n    files). Test rows stay in file order.\n    """\n    numeric = scalar_numeric(pl.scan_parquet(data / "train.parquet").collect_schema())\n    columns = ["feature3", *numeric, *LOW_CATEGORICAL, "rq_size", "rq_items", "rq_item_dup",\n               *[f"rq_{k}_{c}" for c in wide for k in ["rank", "dev", "gap"]]]\n    categories, outputs = None, {}\n    for name, extra in [("train", "watched"), ("test", "id")]:\n        frame = pl.scan_parquet(data / f"{name}.parquet")\n        if rows:\n            frame = frame.filter(pl.col("request_id").hash(7) % rows == 0)\n        if fit_only and name == "train":\n            frame = frame.filter(pl.col("feature1") < CUTOFF)\n        frame = frame.select(extra, "feature3", "request_id", ITEM,\n                             *dict.fromkeys([*LOW_CATEGORICAL, *numeric])).collect()\n        frame = frame.with_columns(context(wide))\n        if categories is None:\n            categories = {c: frame[c].drop_nulls().unique().sort().to_list() for c in LOW_CATEGORICAL}\n        x = np.empty((len(frame), len(columns)), dtype=np.float32)\n        for i, c in enumerate(columns):\n            column = pl.col(c)\n            if c in categories:\n                column = column.cast(pl.Enum(categories[c]), strict=False).to_physical()\n            x[:, i] = frame.select(column.cast(pl.Float32))[c].to_numpy()\n        outputs[name] = (x, frame[extra].to_numpy())\n        del frame\n        gc.collect()\n    categorical = [i for i, c in enumerate(columns) if c in categories]\n    (x_train, y_train), (x_test, test_ids) = outputs["train"], outputs["test"]\n    return x_train, y_train, x_test, test_ids, columns, categorical\n\n\nMODELS = {"xgboost": fit_xgboost, "catboost": fit_catboost, "logreg": fit_logreg}\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--data", type=Path, default=Path("data"))\n    parser.add_argument("--wide-from", type=Path, default=Path("artifacts/local-t026/results.json"))\n    parser.add_argument("--output", type=Path, default=Path("artifacts/local-t029"))\n    parser.add_argument("--models", default="xgboost,logreg,catboost")\n    parser.add_argument("--threads", type=int, default=7)\n    parser.add_argument("--rows", type=int, default=0, help="Keep 1/N of requests for smoke tests")\n    args = parser.parse_args()\n    args.output.mkdir(parents=True, exist_ok=True)\n    start = time.time()\n    wide = json.loads(args.wide_from.read_text())["wide_features"]\n    x_fit, y_fit, x_valid, y_valid, row_ids, columns, categorical = build_matrices(args.data, wide, args.rows)\n    print(f"matrices {x_fit.shape} / {x_valid.shape} in {time.time() - start:.0f}s", flush=True)\n    path = args.output / "results.json"\n    results = json.loads(path.read_text())["results"] if path.exists() else {}\n    for name in args.models.split(","):\n        began = time.time()\n        predictions, extra = MODELS[name](x_fit, y_fit, x_valid, y_valid, categorical, args.threads)\n        auc = float(roc_auc_score(y_valid, predictions))\n        np.savez_compressed(args.output / f"{name}-holdout.npz", row_ids=row_ids,\n                            y_true=y_valid, probabilities=predictions.astype(np.float32))\n        results[name] = {"holdout_auc": auc, "seconds": time.time() - began, **extra}\n        print(name, json.dumps(results[name]), flush=True)\n        path.write_text(json.dumps({\n            "cutoff": CUTOFF.isoformat(), "valid_rows": int(len(y_valid)), "fit_rows": int(len(y_fit)),\n            "features": len(columns), "early_stopping_on_holdout": True, "wide_features": wide,\n            "results": results}, indent=2) + "\\n")\n        gc.collect()\n\n\nif __name__ == "__main__":\n    main()\n', 'xgb_tune.py': '"""T034: XGBoost GPU tuning on the T029 wide request-context features.\n\n`build` writes the fit/holdout matrices once as .npy; `worker` fits a list of named\nconfigs (one process per GPU) and saves holdout predictions with file row ids.\nProtocol matches T029: original cutoff, early stopping on the holdout.\n"""\nimport argparse\nimport json\nfrom pathlib import Path\nimport time\n\nimport numpy as np\nfrom sklearn.metrics import roc_auc_score\n\nBASE = {\n    "objective": "binary:logistic", "eval_metric": "auc", "tree_method": "hist", "device": "cuda",\n    "max_depth": 0, "max_leaves": 127, "grow_policy": "lossguide", "min_child_weight": 20,\n    "eta": 0.05, "subsample": 0.8, "colsample_bytree": 0.5, "lambda": 10, "seed": 42, "max_bin": 127,\n}\n\nCONFIGS = {\n    "t029_gpu": {},\n    "leaves255_eta02": {"max_leaves": 255, "eta": 0.02, "colsample_bytree": 0.3, "min_child_weight": 100},\n    "leaves511_eta02": {"max_leaves": 511, "eta": 0.02, "colsample_bytree": 0.3, "min_child_weight": 200,\n                        "alpha": 1},\n    "leaves63_eta02": {"max_leaves": 63, "eta": 0.02, "colsample_bytree": 0.4, "min_child_weight": 50},\n    "depth10": {"grow_policy": "depthwise", "max_depth": 10, "max_leaves": 0, "eta": 0.03,\n                "colsample_bytree": 0.4, "min_child_weight": 50},\n    "depth12_node": {"grow_policy": "depthwise", "max_depth": 12, "max_leaves": 0, "eta": 0.02,\n                     "colsample_bytree": 0.8, "colsample_bynode": 0.4, "min_child_weight": 100, "gamma": 1},\n    "bin255": {"max_bin": 255, "max_leaves": 255, "eta": 0.02, "colsample_bytree": 0.3,\n               "min_child_weight": 100},\n    "native_cat": {"max_leaves": 255, "eta": 0.02, "colsample_bytree": 0.3, "min_child_weight": 100,\n                   "max_cat_to_onehot": 8, "max_cat_threshold": 64, "_categorical": True},\n    "subsample06": {"max_leaves": 255, "eta": 0.02, "subsample": 0.6, "colsample_bytree": 0.3,\n                    "min_child_weight": 100, "sampling_method": "gradient_based"},\n    "rank_global": {"objective": "rank:pairwise", "max_leaves": 63, "eta": 0.02, "colsample_bytree": 0.4,\n                    "min_child_weight": 50, "lambdarank_pair_method": "mean", "lambdarank_num_pair_per_sample": 8,\n                    "_qid": "none"},\n    "rank_request": {"objective": "rank:pairwise", "max_leaves": 63, "eta": 0.02, "colsample_bytree": 0.4,\n                     "min_child_weight": 50, "lambdarank_pair_method": "mean", "lambdarank_num_pair_per_sample": 8,\n                     "_qid": "request"},\n    "lambda50": {"max_leaves": 255, "eta": 0.02, "colsample_bytree": 0.3, "min_child_weight": 100,\n                 "lambda": 50, "alpha": 5},\n}\n\n\nID_COLUMNS = ["feature5", "feature18", "feature23", "feature20"]\n\n\ndef train_split_columns(data, columns, rows=0):\n    """Pre-cutoff / holdout values of raw train columns, in file order (matches `build_matrices`)."""\n    import polars as pl\n\n    from feature_screen import CUTOFF\n\n    lf = pl.scan_parquet(data / "train.parquet")\n    if rows:\n        lf = lf.filter(pl.col("request_id").hash(7) % rows == 0)\n    frame = lf.select("feature1", *columns).collect()\n    is_valid = (frame["feature1"] >= CUTOFF).to_numpy()\n    return frame.filter(~pl.Series(is_valid)).drop("feature1"), frame.filter(pl.Series(is_valid)).drop("feature1")\n\n\ndef cosine_frame(path, rows=0, fit_only=False):\n    """User x item cosines (T035 `cos` set) plus within-request rank/deviation, in file order."""\n    import polars as pl\n\n    from embed_screen import COS_PAIRS, unit_vectors\n\n    lf = pl.scan_parquet(path)\n    if rows:\n        lf = lf.filter(pl.col("request_id").hash(7) % rows == 0)\n    if fit_only:\n        from feature_screen import CUTOFF\n\n        lf = lf.filter(pl.col("feature1") < CUTOFF)\n    vectors = lf.select("request_id", *dict.fromkeys(c for pair in COS_PAIRS for c in pair)).collect()\n    extra = {}\n    for u, i in COS_PAIRS:\n        extra[f"cos_{u[7:]}_{i[7:]}"] = (unit_vectors(vectors[u]) * unit_vectors(vectors[i])).sum(1)\n    names = list(extra)\n    frame = pl.DataFrame({"request_id": vectors["request_id"], **extra})\n    frame = frame.with_columns([\n        e for c in names for e in (\n            (pl.col(c).rank("average").over("request_id") / pl.len().over("request_id")).alias(f"rq_rank_{c}"),\n            (pl.col(c) - pl.col(c).mean().over("request_id")).alias(f"rq_dev_{c}"))]).drop("request_id")\n    return frame.to_numpy().astype(np.float32), frame.columns\n\n\ndef entity_frame(data, rows=0):\n    """Label-free item/creator features over train+test (file order): upload age, past exposure\n    counts, and the entity\'s mean centered within-request rank of top predictors. Backward-only\n    windows so rows near the end of test are not truncated."""\n    import polars as pl\n\n    def scan(name):\n        lf = pl.scan_parquet(data / f"{name}.parquet")\n        if rows:\n            lf = lf.filter(pl.col("request_id").hash(7) % rows == 0)\n        return lf.select("request_id", "feature1", "feature18", "feature23", "feature58", "feature59",\n                         "feature54", "feature409", "feature506").collect()\n\n    train, test = scan("train"), scan("test")\n    frame = pl.concat([train, test])\n    hours = frame["feature1"].dt.epoch("ms").to_numpy() / 3.6e6\n    columns = {"age_up": hours - frame["feature58"].to_numpy() / 3600.0,\n               "age_up59": hours - frame["feature59"].to_numpy() / 3.6e6}\n    hours = hours - hours.min()\n    size = pl.len().over("request_id")\n    ranks = frame.select([((pl.col(c).rank("average").over("request_id") - (size + 1) / 2) / size).alias(c)\n                          for c in ["feature54", "feature409", "feature506"]])\n    for key in ["feature18", "feature23"]:\n        codes = frame[key].cast(pl.Categorical).to_physical().to_numpy().astype(np.int64)\n        suffix = key[-2:]\n        stamp = codes * 1e4 + hours\n        ordered = np.sort(stamp)\n        columns[f"since_first_{suffix}"] = hours - pl.DataFrame({"k": codes, "t": hours}).select(\n            pl.col("t").min().over("k"))["t"].to_numpy()\n        for window in (1, 6, 24):\n            columns[f"nb{window}_{suffix}"] = (np.searchsorted(ordered, stamp, "left")\n                                               - np.searchsorted(ordered, stamp - window, "left"))\n        means = ranks.with_columns(k=codes).select(pl.all().exclude("k").mean().over("k"))\n        for c in means.columns:\n            columns[f"ent_crk_{c}_{suffix}"] = means[c].to_numpy()\n    values = np.column_stack(list(columns.values())).astype(np.float32)\n    return values[: train.height], values[train.height:], list(columns)\n\n\ndef cosine_columns(data, rows):\n    import polars as pl\n\n    from feature_screen import CUTOFF\n\n    values, names = cosine_frame(data / "train.parquet", rows)\n    lf = pl.scan_parquet(data / "train.parquet")\n    if rows:\n        lf = lf.filter(pl.col("request_id").hash(7) % rows == 0)\n    is_valid = (lf.select("feature1").collect()["feature1"] >= CUTOFF).to_numpy()\n    return values[~is_valid], values[is_valid], names\n\n\ndef build(args):\n    from classic_screen import build_matrices\n\n    start = time.time()\n    wide = json.loads(args.wide_from.read_text())["wide_features"]\n    x_fit, y_fit, x_valid, y_valid, row_ids, columns, categorical = build_matrices(args.data, wide, args.rows)\n    if args.cosines:\n        c_fit, c_valid, names = cosine_columns(args.data, args.rows)\n        x_fit, x_valid, columns = np.hstack([x_fit, c_fit]), np.hstack([x_valid, c_valid]), columns + names\n        del c_fit, c_valid\n    if args.entity:\n        from feature_screen import CUTOFF\n        import polars as pl\n\n        values, _, names = entity_frame(args.data, args.rows)\n        lf = pl.scan_parquet(args.data / "train.parquet")\n        if args.rows:\n            lf = lf.filter(pl.col("request_id").hash(7) % args.rows == 0)\n        is_valid = (lf.select("feature1").collect()["feature1"] >= CUTOFF).to_numpy()\n        x_fit, x_valid = np.hstack([x_fit, values[~is_valid]]), np.hstack([x_valid, values[is_valid]])\n        columns = columns + names\n        del values\n    args.matrices.mkdir(parents=True, exist_ok=True)\n    for name, value in dict(x_fit=x_fit, y_fit=y_fit, x_valid=x_valid, y_valid=y_valid, row_ids=row_ids).items():\n        np.save(args.matrices / f"{name}.npy", value)\n    (args.matrices / "columns.json").write_text(json.dumps({"columns": columns, "categorical": categorical}))\n    print(f"matrices {x_fit.shape} / {x_valid.shape} in {time.time() - start:.0f}s", flush=True)\n\n\ndef worker(args):\n    import xgboost as xgb\n\n    m = args.matrices\n    x_fit, x_valid = np.load(m / "x_fit.npy", mmap_mode="r"), np.load(m / "x_valid.npy", mmap_mode="r")\n    y_fit, y_valid, row_ids = np.load(m / "y_fit.npy"), np.load(m / "y_valid.npy"), np.load(m / "row_ids.npy")\n    meta = json.loads((m / "columns.json").read_text())\n    feature_types = ["c" if i in set(meta["categorical"]) else "q" for i in range(len(meta["columns"]))]\n    args.output.mkdir(parents=True, exist_ok=True)\n    for name in args.configs.split(","):\n        began = time.time()\n        params = {**BASE, **CONFIGS[name], "device": args.device}\n        categorical = params.pop("_categorical", False)\n        qid = params.pop("_qid", None)\n        types = dict(feature_types=feature_types, enable_categorical=True) if categorical else {}\n        if qid == "request":\n            fit_rq, _ = train_split_columns(args.data, ["request_id"], args.rows)\n            codes = fit_rq["request_id"].to_numpy()\n            types["qid"] = np.cumsum(np.r_[True, codes[1:] != codes[:-1]]).astype(np.uint32)\n        elif qid == "none":\n            types["qid"] = np.zeros(len(y_fit), dtype=np.uint32)\n        fit = xgb.QuantileDMatrix(np.asarray(x_fit), y_fit, max_bin=params["max_bin"], **types)\n        types.pop("qid", None)\n        valid = xgb.QuantileDMatrix(np.asarray(x_valid), y_valid, ref=fit, max_bin=params["max_bin"], **types)\n        model = xgb.train(params, fit, args.rounds, evals=[(valid, "holdout")],\n                          early_stopping_rounds=300, verbose_eval=500)\n        predictions = model.predict(valid, iteration_range=(0, model.best_iteration + 1))\n        if qid:\n            predictions = 1 / (1 + np.exp(-predictions))\n        auc = float(roc_auc_score(y_valid, predictions))\n        np.savez_compressed(args.output / f"{name}-holdout.npz", row_ids=row_ids, y_true=y_valid,\n                            probabilities=predictions.astype(np.float32))\n        gain = model.get_score(importance_type="total_gain")\n        top = sorted(gain.items(), key=lambda kv: -kv[1])[:30]\n        result = {"holdout_auc": auc, "best_iteration": int(model.best_iteration),\n                  "seconds": time.time() - began, "params": params,\n                  "top_gain": [[meta["columns"][int(k[1:])], v] for k, v in top]}\n        (args.output / f"{name}.json").write_text(json.dumps(result, indent=2) + "\\n")\n        print(name, json.dumps({k: result[k] for k in ["holdout_auc", "best_iteration", "seconds"]}), flush=True)\n        del fit, valid, model\n\n\ndef catboost_worker(args):\n    """CatBoost (GPU) on the same matrices; low-cardinality codes and optional raw IDs as categoricals."""\n    from catboost import CatBoostClassifier, FeaturesData, Pool\n\n    m = args.matrices\n    meta = json.loads((m / "columns.json").read_text())\n    y_fit, y_valid, row_ids = np.load(m / "y_fit.npy"), np.load(m / "y_valid.npy"), np.load(m / "row_ids.npy")\n    categorical = meta["categorical"]\n    dense = [i for i in range(len(meta["columns"])) if i not in set(categorical)]\n    ids = ID_COLUMNS if args.catboost_ids else []\n    raw = train_split_columns(args.data, ids, args.rows) if ids else None\n\n    def pool(split, y, part):\n        x = np.load(m / f"x_{split}.npy", mmap_mode="r")\n        codes = np.nan_to_num(np.asarray(x[:, categorical]), nan=-1).astype(np.int64).astype(str)\n        if ids:\n            codes = np.hstack([codes, part.fill_null("__NA__").to_numpy().astype(str)])\n        return Pool(FeaturesData(num_feature_data=np.ascontiguousarray(x[:, dense], dtype=np.float32),\n                                 cat_feature_data=codes.astype(object)), y)\n\n    began = time.time()\n    fit = pool("fit", y_fit, raw[0] if ids else None)\n    valid = pool("valid", y_valid, raw[1] if ids else None)\n    name = "catboost_ids" if ids else "catboost"\n    model = CatBoostClassifier(\n        iterations=args.rounds, learning_rate=0.06, depth=8, border_count=128, l2_leaf_reg=10,\n        eval_metric="AUC", metric_period=50, od_type="Iter", od_wait=300, random_seed=42, verbose=500,\n        use_best_model=True, task_type="GPU" if args.device.startswith("cuda") else "CPU",\n        **({"devices": "0"} if args.device.startswith("cuda") else {"thread_count": 6}),\n        max_ctr_complexity=2, gpu_ram_part=0.9 if args.device.startswith("cuda") else None,\n    )\n    model.fit(fit, eval_set=valid)\n    predictions = model.predict_proba(valid)[:, 1]\n    auc = float(roc_auc_score(y_valid, predictions))\n    args.output.mkdir(parents=True, exist_ok=True)\n    np.savez_compressed(args.output / f"{name}-holdout.npz", row_ids=row_ids, y_true=y_valid,\n                        probabilities=predictions.astype(np.float32))\n    model.save_model(str(args.output / f"{name}.cbm"))\n    result = {"holdout_auc": auc, "best_iteration": int(model.get_best_iteration()),\n              "seconds": time.time() - began, "ids": ids}\n    (args.output / f"{name}.json").write_text(json.dumps(result, indent=2) + "\\n")\n    print(name, json.dumps(result), flush=True)\n\n\ndef catboost_refit(args):\n    """Fixed-iteration CatBoost (+raw IDs) on every labeled row; `--configs seed:iterations,...`."""\n    import polars as pl\n    from catboost import CatBoostClassifier, FeaturesData, Pool\n\n    m = args.matrices\n    meta = json.loads((m / "columns.json").read_text())\n    y_train, test_ids = np.load(m / "y_train.npy"), np.load(m / "test_ids.npy")\n    categorical = meta["categorical"]\n    dense = [i for i in range(len(meta["columns"])) if i not in set(categorical)]\n\n    def raw(name, extra):\n        lf = pl.scan_parquet(args.data / f"{name}.parquet")\n        if args.rows:\n            lf = lf.filter(pl.col("request_id").hash(7) % args.rows == 0)\n        return lf.select(extra, *ID_COLUMNS).collect()\n\n    def pool(split, ids, y=None):\n        x = np.load(m / f"x_{split}.npy", mmap_mode="r")\n        codes = np.nan_to_num(np.asarray(x[:, categorical]), nan=-1).astype(np.int64).astype(str)\n        codes = np.hstack([codes, ids.select(ID_COLUMNS).fill_null("__NA__").to_numpy().astype(str)])\n        return Pool(FeaturesData(num_feature_data=np.ascontiguousarray(x[:, dense], dtype=np.float32),\n                                 cat_feature_data=codes.astype(object)), y)\n\n    train_ids, test_raw = raw("train", "watched"), raw("test", "id")\n    if not np.array_equal(train_ids["watched"].to_numpy(), y_train):\n        raise ValueError("Train ID columns are not aligned with the matrices")\n    if not np.array_equal(test_raw["id"].to_numpy(), test_ids):\n        raise ValueError("Test ID columns are not aligned with the matrices")\n    template = None\n    if (args.data / "sample_submission.csv").is_file() and not args.rows:\n        template = pl.read_csv(args.data / "sample_submission.csv")\n        if not np.array_equal(template["id"].to_numpy(), test_ids):\n            raise ValueError("Test ids differ from the submission template")\n    train, test = pool("train", train_ids, y_train), pool("test", test_raw)\n    args.output.mkdir(parents=True, exist_ok=True)\n    gpu = args.device.startswith("cuda")\n    for item in args.configs.split(","):\n        began = time.time()\n        seed, iterations = (int(v) for v in item.split(":"))\n        tag = f"catboost_ids-s{seed}"\n        model = CatBoostClassifier(\n            iterations=iterations, learning_rate=0.06, depth=8, border_count=128, l2_leaf_reg=10,\n            random_seed=seed, verbose=500, task_type="GPU" if gpu else "CPU",\n            **({"devices": "0", "gpu_ram_part": 0.9} if gpu else {"thread_count": 6}), max_ctr_complexity=2)\n        model.fit(train)\n        predictions = model.predict_proba(test)[:, 1].astype(np.float32)\n        if not np.isfinite(predictions).all():\n            raise ValueError(f"Nonfinite predictions: {tag}")\n        np.savez_compressed(args.output / f"{tag}-test.npz", ids=test_ids, probabilities=predictions)\n        model.save_model(str(args.output / f"{tag}.cbm"))\n        if template is not None:\n            template.with_columns(pl.Series("watched", predictions)).write_csv(args.output / f"submission-{tag}.csv")\n        result = {"iterations": iterations, "seed": seed, "training_rows": int(len(y_train)),\n                  "ids": ID_COLUMNS, "seconds": time.time() - began}\n        (args.output / f"{tag}-refit.json").write_text(json.dumps(result, indent=2) + "\\n")\n        print(tag, json.dumps(result), flush=True)\n        del model\n\n\ndef kfold(args):\n    """5-fold (grouped by request) fits on every labeled row with in-fold early stopping.\n\n    `--configs` takes XGBoost config names and/or `catboost_ids`. Saves out-of-fold\n    predictions on the train rows plus per-fold and fold-mean test predictions.\n    """\n    import polars as pl\n    from sklearn.model_selection import GroupKFold\n\n    m = args.matrices\n    meta = json.loads((m / "columns.json").read_text())\n    y_train, test_ids = np.load(m / "y_train.npy"), np.load(m / "test_ids.npy")\n    lf = pl.scan_parquet(args.data / "train.parquet")\n    test_lf = pl.scan_parquet(args.data / "test.parquet")\n    if args.rows:\n        lf = lf.filter(pl.col("request_id").hash(7) % args.rows == 0)\n        test_lf = test_lf.filter(pl.col("request_id").hash(7) % args.rows == 0)\n    train_raw = lf.select("watched", "request_id", *ID_COLUMNS).collect()\n    if not np.array_equal(train_raw["watched"].to_numpy(), y_train):\n        raise ValueError("Train rows are not aligned with the matrices")\n    folds = list(GroupKFold(args.folds).split(y_train, groups=train_raw["request_id"].to_numpy()))\n    x_train, x_test = np.load(m / "x_train.npy", mmap_mode="r"), np.load(m / "x_test.npy", mmap_mode="r")\n    template = None\n    if (args.data / "sample_submission.csv").is_file() and not args.rows:\n        template = pl.read_csv(args.data / "sample_submission.csv")\n        if not np.array_equal(template["id"].to_numpy(), test_ids):\n            raise ValueError("Test ids differ from the submission template")\n    args.output.mkdir(parents=True, exist_ok=True)\n    for name in args.configs.split(","):\n        began = time.time()\n        oof = np.full(len(y_train), np.nan, dtype=np.float32)\n        tests, info = [], []\n        if name == "catboost_ids":\n            from catboost import CatBoostClassifier, FeaturesData, Pool\n\n            test_raw = test_lf.select("id", *ID_COLUMNS).collect()\n            if not np.array_equal(test_raw["id"].to_numpy(), test_ids):\n                raise ValueError("Test rows are not aligned with the matrices")\n            categorical = meta["categorical"]\n            dense = [i for i in range(len(meta["columns"])) if i not in set(categorical)]\n\n            def codes(x, raw):\n                low = np.nan_to_num(np.asarray(x[:, categorical]), nan=-1).astype(np.int64).astype(str)\n                return np.hstack([low, raw.select(ID_COLUMNS).fill_null("__NA__").to_numpy().astype(str)]).astype(object)\n\n            train_codes, test_codes = codes(x_train, train_raw), codes(x_test, test_raw)\n            test_pool = Pool(FeaturesData(num_feature_data=np.ascontiguousarray(x_test[:, dense], dtype=np.float32),\n                                          cat_feature_data=test_codes))\n            gpu = args.device.startswith("cuda")\n            for k, (a, v) in enumerate(folds):\n                def pool(rows):\n                    return Pool(FeaturesData(num_feature_data=np.ascontiguousarray(x_train[rows][:, dense], dtype=np.float32),\n                                             cat_feature_data=train_codes[rows]), y_train[rows])\n                model = CatBoostClassifier(\n                    iterations=args.rounds, learning_rate=0.06, depth=8, border_count=128, l2_leaf_reg=10,\n                    eval_metric="AUC", metric_period=50, od_type="Iter", od_wait=300, random_seed=42 + k,\n                    verbose=500, use_best_model=True, task_type="GPU" if gpu else "CPU",\n                    **({"devices": "0", "gpu_ram_part": 0.9} if gpu else {"thread_count": 6}), max_ctr_complexity=2)\n                fit_pool, valid_pool = pool(a), pool(v)\n                model.fit(fit_pool, eval_set=valid_pool)\n                oof[v] = model.predict_proba(valid_pool)[:, 1]\n                tests.append(model.predict_proba(test_pool)[:, 1].astype(np.float32))\n                info.append({"fold": k, "best_iteration": int(model.get_best_iteration()),\n                             "fold_auc": float(roc_auc_score(y_train[v], oof[v])), "seconds": time.time() - began})\n                print(name, json.dumps(info[-1]), flush=True)\n                del model, fit_pool, valid_pool\n        else:\n            import xgboost as xgb\n\n            params = {**BASE, **CONFIGS[name], "device": args.device}\n            if params.pop("_categorical", False) or params.pop("_qid", None):\n                raise ValueError("Only plain XGBoost configs are supported in kfold")\n            for k, (a, v) in enumerate(folds):\n                params["seed"] = 42 + k\n                fit = xgb.QuantileDMatrix(x_train[a], y_train[a], max_bin=params["max_bin"])\n                valid = xgb.QuantileDMatrix(x_train[v], y_train[v], max_bin=params["max_bin"], ref=fit)\n                model = xgb.train(params, fit, args.rounds, evals=[(valid, "fold")],\n                                  early_stopping_rounds=300, verbose_eval=500)\n                best = model.best_iteration\n                oof[v] = model.predict(valid, iteration_range=(0, best + 1))\n                test = xgb.QuantileDMatrix(x_test, max_bin=params["max_bin"], ref=fit)\n                tests.append(model.predict(test, iteration_range=(0, best + 1)).astype(np.float32))\n                info.append({"fold": k, "best_iteration": int(best),\n                             "fold_auc": float(roc_auc_score(y_train[v], oof[v])), "seconds": time.time() - began})\n                print(name, json.dumps(info[-1]), flush=True)\n                del fit, valid, test, model\n        mean = np.mean(tests, 0).astype(np.float32)\n        if not np.isfinite(mean).all() or np.isnan(oof).any():\n            raise ValueError(f"Incomplete or nonfinite predictions: {name}")\n        np.savez_compressed(args.output / f"{name}-kfold.npz", ids=test_ids, probabilities=mean,\n                            fold_probabilities=np.stack(tests), oof=oof, y_train=y_train)\n        if template is not None:\n            template.with_columns(pl.Series("watched", mean)).write_csv(args.output / f"submission-{name}-kfold.csv")\n        result = {"folds": info, "oof_auc": float(roc_auc_score(y_train, oof)), "seconds": time.time() - began}\n        (args.output / f"{name}-kfold.json").write_text(json.dumps(result, indent=2) + "\\n")\n        print(name, "oof", result["oof_auc"], flush=True)\n\n\ndef build_full(args):\n    from classic_screen import full_matrices\n\n    start = time.time()\n    wide = json.loads(args.wide_from.read_text())["wide_features"]\n    x_train, y_train, x_test, test_ids, columns, categorical = full_matrices(args.data, wide, args.rows, args.fit_only)\n    if args.cosines:\n        c_train, names = cosine_frame(args.data / "train.parquet", args.rows, args.fit_only)\n        x_train, columns = np.hstack([x_train, c_train]), columns + names\n        del c_train\n        c_test, _ = cosine_frame(args.data / "test.parquet", args.rows)\n        x_test = np.hstack([x_test, c_test])\n        del c_test\n    args.matrices.mkdir(parents=True, exist_ok=True)\n    for name, value in dict(x_train=x_train, y_train=y_train, x_test=x_test, test_ids=test_ids).items():\n        np.save(args.matrices / f"{name}.npy", value)\n    (args.matrices / "columns.json").write_text(json.dumps({"columns": columns, "categorical": categorical}))\n    print(f"full matrices {x_train.shape} / {x_test.shape} in {time.time() - start:.0f}s", flush=True)\n\n\ndef refit(args):\n    """Fixed-round fits on every labeled row; `--configs name[@seed]:rounds,...`; test predictions saved."""\n    import polars as pl\n    import xgboost as xgb\n\n    m = args.matrices\n    y_train, test_ids = np.load(m / "y_train.npy"), np.load(m / "test_ids.npy")\n    meta = json.loads((m / "columns.json").read_text())\n    args.output.mkdir(parents=True, exist_ok=True)\n    template = None\n    if (args.data / "sample_submission.csv").is_file() and not args.rows:\n        template = pl.read_csv(args.data / "sample_submission.csv")\n        if not np.array_equal(template["id"].to_numpy(), test_ids):\n            raise ValueError("Test ids differ from the submission template")\n    for item in args.configs.split(","):\n        began = time.time()\n        spec, rounds = item.split(":")\n        name, _, seed = spec.partition("@")\n        tag = f"{name}-s{seed}" if seed else name\n        params = {**BASE, **CONFIGS[name], "device": args.device}\n        if seed:\n            params["seed"] = int(seed)\n        if params.pop("_categorical", False):\n            raise ValueError("Native categorical configs are not supported in refit")\n        train = xgb.QuantileDMatrix(np.load(m / "x_train.npy", mmap_mode="r"), y_train, max_bin=params["max_bin"])\n        model = xgb.train(params, train, int(rounds))\n        test = xgb.QuantileDMatrix(np.load(m / "x_test.npy", mmap_mode="r"), max_bin=params["max_bin"], ref=train)\n        predictions = model.predict(test).astype(np.float32)\n        del train, test\n        if not np.isfinite(predictions).all():\n            raise ValueError(f"Nonfinite predictions: {tag}")\n        np.savez_compressed(args.output / f"{tag}-test.npz", ids=test_ids, probabilities=predictions)\n        model.save_model(args.output / f"{tag}.ubj")\n        if template is not None:\n            template.with_columns(pl.Series("watched", predictions)).write_csv(args.output / f"submission-{tag}.csv")\n        result = {"rounds": int(rounds), "training_rows": int(len(y_train)), "features": len(meta["columns"]),\n                  "seconds": time.time() - began, "params": params}\n        (args.output / f"{tag}-refit.json").write_text(json.dumps(result, indent=2) + "\\n")\n        print(tag, json.dumps({k: result[k] for k in ["rounds", "training_rows", "seconds"]}), flush=True)\n        del model\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("mode", choices=["build", "worker", "build_full", "refit", "catboost", "catboost_refit", "kfold"])\n    parser.add_argument("--data", type=Path, default=Path("data"))\n    parser.add_argument("--wide-from", type=Path, default=Path("artifacts/local-t026/results.json"))\n    parser.add_argument("--matrices", type=Path, default=Path("artifacts/local-t034/matrices"))\n    parser.add_argument("--output", type=Path, default=Path("artifacts/local-t034"))\n    parser.add_argument("--configs", default=",".join(CONFIGS))\n    parser.add_argument("--device", default="cuda")\n    parser.add_argument("--cosines", action="store_true", help="Append T035 user x item cosine features")\n    parser.add_argument("--entity", action="store_true", help="build: append label-free item/creator features")\n    parser.add_argument("--rounds", type=int, default=20000)\n    parser.add_argument("--catboost-ids", action="store_true", help="catboost: add raw user/item/ID columns")\n    parser.add_argument("--fit-only", action="store_true", help="build_full: train rows before the cutoff only")\n    parser.add_argument("--folds", type=int, default=5)\n    parser.add_argument("--rows", type=int, default=0, help="Keep 1/N of requests for smoke tests")\n    args = parser.parse_args()\n    {"build": build, "worker": worker, "build_full": build_full, "refit": refit,\n     "catboost": catboost_worker, "catboost_refit": catboost_refit, "kfold": kfold}[args.mode](args)\n\n\nif __name__ == "__main__":\n    main()\n', 'embed_screen.py': '"""T035 local LightGBM screen: item-embedding request context and similar-video watch rates.\n\nRow-level vectors feature40-46 are near-constant per item (cos 0.999 to the item mean),\nso each item gets a label-free mean embedding from train+test. Variants on top of the\nT023/T031 request-context control:\n- cos: user(26-32) x item(40-46) cosines plus their within-request rank/deviation;\n- embctx: cosine to the leave-one-out request mean item embedding, request embedding\n  diversity, distinct creators/categories in the request;\n- knn: out-of-fold watch rate of the top-k most similar other items (embedding kNN).\nScores the original 307,528-row holdout; request_id is only used for grouping.\n"""\nimport argparse\nimport gc\nimport json\nfrom pathlib import Path\nimport time\n\nimport lightgbm as lgb\nimport numpy as np\nimport polars as pl\nfrom sklearn.metrics import roc_auc_score\nfrom sklearn.model_selection import GroupKFold\n\nfrom cf_screen import within_request_auc\nfrom feature_screen import CUTOFF, ITEM, LOW_CATEGORICAL, RANKED, request_features, scalar_numeric\n\nUSER_VECTORS = [f"feature{i}" for i in range(26, 33)]\nITEM_VECTORS = [f"feature{i}" for i in range(40, 47)]\nCOS_PAIRS = [(u, i) for u, i in zip(USER_VECTORS, ITEM_VECTORS)] + [\n    ("feature32", "feature43"), ("feature30", "feature46"), ("feature31", "feature44")]\nKNN_VECTORS = ["feature40", "feature43", "feature46"]\nCREATOR, CATEGORY = "feature23", "feature19"\nSMOOTHING = 20.0\n\n\ndef unit_vectors(series):\n    values = series.fill_null([0.0] * 32).list.to_array(32).to_numpy().astype(np.float32)\n    return values / np.maximum(np.linalg.norm(values, axis=1, keepdims=True), 1e-6)\n\n\ndef item_neighbours(data, rows, k, chunk=512):\n    """Label-free item mean embeddings over train+test and their top-k cosine neighbours."""\n    parts = []\n    for name in ["train.parquet", "test.parquet"]:\n        lf = pl.scan_parquet(data / name)\n        if rows:\n            lf = lf.filter(pl.col("request_id").hash(7) % rows == 0)\n        parts.append(lf.select(ITEM, *KNN_VECTORS).collect())\n    frame = pl.concat(parts)\n    del parts\n    blocks = [unit_vectors(frame[c]) for c in KNN_VECTORS]\n    keys, inverse = np.unique(frame[ITEM].to_numpy(), return_inverse=True)\n    counts = np.bincount(inverse).astype(np.float32)\n    means = []\n    for block in blocks:\n        total = np.zeros((len(keys), 32), dtype=np.float64)\n        np.add.at(total, inverse, block)\n        mean = total / counts[:, None]\n        means.append(mean / np.maximum(np.linalg.norm(mean, axis=1, keepdims=True), 1e-6))\n    embedding = (np.hstack(means) / np.sqrt(len(means))).astype(np.float32)\n    del frame, blocks\n    k = min(k, len(keys) - 1)\n    index = np.empty((len(keys), k), dtype=np.int32)\n    weight = np.empty((len(keys), k), dtype=np.float32)\n    for start in range(0, len(keys), chunk):\n        sims = embedding[start:start + chunk] @ embedding.T\n        sims[np.arange(len(sims)), np.arange(start, start + len(sims))] = -np.inf\n        top = np.argpartition(-sims, k, axis=1)[:, :k]\n        index[start:start + chunk] = top\n        weight[start:start + chunk] = np.take_along_axis(sims, top, axis=1)\n    return pl.DataFrame({ITEM: keys}).with_row_index("item_idx"), index, weight, embedding\n\n\ndef knn_rates(item_idx, index, weight, s, n, prior, temperature=0.05):\n    """Similarity-weighted neighbour watch rate (softmax over cosine) and its support."""\n    w = np.exp((weight[item_idx] - weight[item_idx].max(axis=1, keepdims=True)) / temperature)\n    ns, nn = n[index[item_idx]], s[index[item_idx]]\n    rate = ((w * nn).sum(1) + SMOOTHING * prior) / ((w * ns).sum(1) + SMOOTHING)\n    support = (w * ns).sum(1) / w.sum(1)\n    seen = ns > 0\n    best = np.where(seen, weight[item_idx], -np.inf).max(1)\n    return rate, np.log1p(support), np.where(np.isfinite(best), best, np.nan)\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--data", type=Path, default=Path("data"))\n    parser.add_argument("--output", type=Path, default=Path("artifacts/local-t035"))\n    parser.add_argument("--variants", default="control,cos,embctx,knn")\n    parser.add_argument("--neighbours", type=int, default=50)\n    parser.add_argument("--learning-rate", type=float, default=0.05)\n    parser.add_argument("--rounds", type=int, default=4000)\n    parser.add_argument("--threads", type=int, default=8)\n    parser.add_argument("--rows", type=int, default=0, help="Keep 1/N of requests for smoke tests")\n    args = parser.parse_args()\n    args.output.mkdir(parents=True, exist_ok=True)\n    start = time.time()\n    items, index, weight, embedding = item_neighbours(args.data, args.rows, args.neighbours)\n    print(f"item kNN ready ({len(items)} items) {time.time() - start:.0f}s", flush=True)\n\n    schema = pl.scan_parquet(args.data / "train.parquet").collect_schema()\n    numeric = scalar_numeric(schema)\n    lf = pl.scan_parquet(args.data / "train.parquet").with_row_index("row")\n    if args.rows:\n        lf = lf.filter(pl.col("request_id").hash(7) % args.rows == 0)\n    keys = sorted(set([ITEM, CREATOR, CATEGORY] + LOW_CATEGORICAL), key=lambda c: int(c[7:]))\n    train = lf.select("row", "watched", "request_id", "feature1", "feature3", *keys,\n                      *[c for c in numeric if c not in keys]).collect()\n    rq, rq_names = request_features(train.select("row", "request_id", ITEM, *RANKED))\n    train = train.join(rq, on="row", how="left")\n    del rq\n    train = train.join(items.with_columns(pl.col(ITEM).cast(train.schema[ITEM])), on=ITEM, how="left")\n    extra = {}\n\n    # cos: user x item cosine similarities and their within-request rank/deviation.\n    vectors = lf.select(*dict.fromkeys(c for pair in COS_PAIRS for c in pair)).collect()\n    units = {c: unit_vectors(vectors[c]) for c in vectors.columns}\n    del vectors\n    cos_names = []\n    for u, i in COS_PAIRS:\n        name = f"cos_{u[7:]}_{i[7:]}"\n        extra[name] = (units[u] * units[i]).sum(1)\n        cos_names.append(name)\n    frame = pl.DataFrame({"request_id": train["request_id"], **extra})\n    ranked = frame.select([\n        e for c in cos_names for e in (\n            (pl.col(c).rank("average").over("request_id") / pl.len().over("request_id")).alias(f"rq_rank_{c}"),\n            (pl.col(c) - pl.col(c).mean().over("request_id")).alias(f"rq_dev_{c}"))])\n    for c in ranked.columns:\n        extra[c] = ranked[c].to_numpy()\n    cos_names += ranked.columns\n    del ranked, frame\n\n    # embctx: similarity to the rest of the request in item-embedding space.\n    requests = train["request_id"].to_numpy()\n    _, request_code = np.unique(requests, return_inverse=True)\n    size = np.bincount(request_code).astype(np.float32)[request_code]\n    ctx_names = []\n    for c in ["feature40", "feature43", "feature46"]:\n        u = units.get(c)\n        if u is None:\n            u = unit_vectors(lf.select(c).collect()[c])\n        total = np.zeros((request_code.max() + 1, 32), dtype=np.float64)\n        np.add.at(total, request_code, u)\n        others = total[request_code] - u\n        norm = np.linalg.norm(others, axis=1)\n        loo = np.where(size > 1, (u * others).sum(1) / np.maximum(norm, 1e-6), np.nan)\n        sq = (total ** 2).sum(1)[request_code]\n        diversity = np.where(size > 1, (sq - size) / np.maximum(size * (size - 1), 1), np.nan)\n        extra[f"emb_loo_{c}"], extra[f"emb_div_{c}"] = loo.astype(np.float32), diversity.astype(np.float32)\n        extra[f"emb_lift_{c}"] = (loo - diversity).astype(np.float32)\n        ctx_names += [f"emb_loo_{c}", f"emb_div_{c}", f"emb_lift_{c}"]\n    del units\n    counts = train.select(\n        pl.col(CREATOR).n_unique().over("request_id").alias("rq_creators"),\n        pl.col(CATEGORY).n_unique().over("request_id").alias("rq_categories"),\n        pl.len().over(["request_id", CREATOR]).alias("rq_creator_dup"),\n        pl.len().over(["request_id", CATEGORY]).alias("rq_category_dup"))\n    for c in counts.columns:\n        extra[c] = counts[c].to_numpy().astype(np.float32)\n    extra["rq_creator_share"] = extra["rq_creator_dup"] / size\n    ctx_names += [*counts.columns, "rq_creator_share"]\n    del counts\n\n    # knn: out-of-fold neighbour watch rates (neighbour lists exclude the item itself).\n    is_valid = (train["feature1"] >= CUTOFF).to_numpy()\n    y = train["watched"].to_numpy().astype(np.float64)\n    item_idx = train["item_idx"].to_numpy().astype(np.int64)\n    fit_idx, valid_idx = np.flatnonzero(~is_valid), np.flatnonzero(is_valid)\n    prior = float(y[fit_idx].mean())\n    knn_names = ["knn_rate", "knn_support", "knn_best_sim", "item_rate", "item_support", "knn_minus_item"]\n    for name in knn_names:\n        extra[name] = np.full(len(train), np.nan, dtype=np.float32)\n\n    def fill(source, target):\n        s = np.bincount(item_idx[source], y[source], minlength=len(items))\n        n = np.bincount(item_idx[source], minlength=len(items)).astype(np.float64)\n        rate, support, best = knn_rates(item_idx[target], index, weight, s, n, prior)\n        own = (s[item_idx[target]] + SMOOTHING * prior) / (n[item_idx[target]] + SMOOTHING)\n        for name, value in zip(knn_names, [rate, support, best, own, np.log1p(n[item_idx[target]]), rate - own]):\n            extra[name][target] = value\n    groups = train["request_id"].to_numpy()[fit_idx]\n    for a, b in GroupKFold(5).split(fit_idx, groups=groups):\n        fill(fit_idx[a], fit_idx[b])\n    fill(fit_idx, valid_idx)\n    print(f"features ready {time.time() - start:.0f}s", flush=True)\n\n    categories = {c: train.filter(pl.Series(~is_valid))[c].drop_nulls().unique().sort() for c in LOW_CATEGORICAL}\n    base = ["feature3", *[c for c in numeric if c not in categories], *LOW_CATEGORICAL, *rq_names]\n    groups_ = {"control": base, "cos": base + cos_names, "embctx": base + cos_names + ctx_names,\n               "knn": base + cos_names + ctx_names + knn_names}\n    y_fit, y_valid = y[fit_idx], y[valid_idx]\n    univariate = {}\n    for name in cos_names[:10] + ctx_names + knn_names:\n        v = extra[name][valid_idx]\n        ok = np.isfinite(v)\n        auc = roc_auc_score(y_valid[ok], v[ok])\n        univariate[name] = round(float(max(auc, 1 - auc)), 4)\n    print("univariate holdout AUC (max of +/-):", json.dumps(univariate), flush=True)\n    item_seen = np.bincount(item_idx[fit_idx], minlength=len(items))[item_idx[valid_idx]] > 0\n\n    def matrix(rows, columns):\n        out = np.empty((len(rows), len(columns)), dtype=np.float32)\n        for j, c in enumerate(columns):\n            if c in extra:\n                out[:, j] = extra[c][rows]\n            elif c in categories:\n                out[:, j] = train.select(pl.col(c).cast(pl.Enum(categories[c].to_list()), strict=False)\n                                         .to_physical().cast(pl.Float32))[c].to_numpy()[rows]\n            else:\n                out[:, j] = train[c].cast(pl.Float32).to_numpy()[rows]\n        return out\n\n    results = {}\n    valid_requests = train["request_id"][valid_idx]\n    for name in args.variants.split(","):\n        columns = groups_[name]\n        began = time.time()\n        x_fit, x_valid = matrix(fit_idx, columns), matrix(valid_idx, columns)\n        categorical = [i for i, c in enumerate(columns) if c in categories]\n        train_set = lgb.Dataset(x_fit, y_fit, feature_name=columns, categorical_feature=categorical, free_raw_data=True)\n        valid_set = lgb.Dataset(x_valid, y_valid, reference=train_set)\n        params = {\n            "objective": "binary", "metric": "auc", "learning_rate": args.learning_rate,\n            "num_leaves": 127, "min_data_in_leaf": 200, "feature_fraction": 0.5,\n            "bagging_fraction": 0.8, "bagging_freq": 1, "lambda_l2": 10,\n            "max_bin": 127, "seed": 42, "num_threads": args.threads, "verbosity": -1,\n            "cat_smooth": 20, "max_cat_to_onehot": 8,\n        }\n        model = lgb.train(params, train_set, args.rounds, valid_sets=[valid_set], valid_names=["holdout"],\n                          callbacks=[lgb.early_stopping(200, verbose=False), lgb.log_evaluation(250)])\n        prediction = model.predict(x_valid, num_iteration=model.best_iteration)\n        importance = sorted(zip(columns, model.feature_importance("gain")), key=lambda p: -p[1])[:30]\n        results[name] = {"holdout_auc": float(roc_auc_score(y_valid, prediction)),\n                         "within_request_auc": within_request_auc(valid_requests, y_valid, prediction),\n                         "seen_item_auc": float(roc_auc_score(y_valid[item_seen], prediction[item_seen])),\n                         "unseen_item_auc": float(roc_auc_score(y_valid[~item_seen], prediction[~item_seen])),\n                         "best_iteration": model.best_iteration, "features": len(columns),\n                         "seconds": time.time() - began, "top_gain": [[c, float(g)] for c, g in importance]}\n        np.save(args.output / f"valid-{name}.npy", prediction.astype(np.float32))\n        print(name, json.dumps({k: v for k, v in results[name].items() if k != "top_gain"}), flush=True)\n        print(" top:", [c for c, _ in importance[:15]], flush=True)\n        (args.output / "results.json").write_text(json.dumps({\n            "cutoff": CUTOFF.isoformat(), "valid_rows": int(len(y_valid)), "fit_rows": int(len(y_fit)),\n            "items": len(items), "neighbours": int(index.shape[1]), "cos_pairs": COS_PAIRS,\n            "univariate": univariate, "early_stopping_on_holdout": True, "results": results,\n            "seconds": time.time() - start}, indent=2) + "\\n")\n        del x_fit, x_valid, train_set, valid_set, model\n        gc.collect()\n    np.save(args.output / "valid-row-ids.npy", train["row"].to_numpy()[valid_idx])\n\n\nif __name__ == "__main__":\n    main()\n', 'cf_screen.py': '"""T031 local LightGBM screen of user-cluster collaborative-filtering target encodings.\n\nUsers are almost always new, so "similar users" are defined by clustering request-level\nprofiles (request-constant scalar columns, dates excluded) and the feature32 user vector\nover train+test requests without labels. Out-of-fold (cluster x item/creator/category)\nwatch rates are then added to the T023 request-context control. Scores the original\n307,528-row holdout; request_id is only used for grouping.\n"""\nimport argparse\nimport gc\nimport json\nfrom pathlib import Path\nimport time\n\nimport lightgbm as lgb\nimport numpy as np\nimport polars as pl\nfrom sklearn.cluster import MiniBatchKMeans\nfrom sklearn.decomposition import PCA\nfrom sklearn.metrics import roc_auc_score\nfrom sklearn.model_selection import GroupKFold\n\nfrom feature_screen import CUTOFF, ITEM, LOW_CATEGORICAL, RANKED, request_features, scalar_numeric, target_encode\n\nDATES = {"feature1", "feature2", "feature3", "feature4"}\nUSER_VECTOR = "feature32"\nITEM_KEYS = [ITEM, "feature23", "feature19"]\nCROSSES = [("pc64", ITEM), ("pc512", ITEM), ("pc512", "feature23"), ("pc64", "feature19"),\n           ("uv256", ITEM), ("uv256", "feature23")]\n\n\ndef request_constant(frame, columns):\n    first = frame.filter(pl.col("request_id").is_first_distinct())\n    counts = frame.group_by("request_id").agg([pl.col(c).n_unique() for c in columns])\n    return [c for c in columns if float((counts[c] <= 1).mean()) > 0.99], first\n\n\ndef clusters(data, rows, seed=42):\n    """Label-free request clusters over train+test."""\n    schema = pl.scan_parquet(data / "train.parquet").collect_schema()\n    numeric = [c for c in scalar_numeric(schema) if c not in DATES]\n\n    def scan(name):\n        lf = pl.scan_parquet(data / name)\n        if rows:\n            lf = lf.filter(pl.col("request_id").hash(7) % rows == 0)\n        return lf\n    probe = scan("train.parquet").select("request_id", *numeric).head(200_000).collect()\n    profile, _ = request_constant(probe, numeric)\n    del probe\n    parts = [scan(n).filter(pl.col("request_id").is_first_distinct())\n             .select("request_id", *profile, pl.col(USER_VECTOR)).collect() for n in ["train.parquet", "test.parquet"]]\n    req = pl.concat(parts)\n    del parts\n    ranked = req.select([(pl.col(c).rank("average") / pl.col(c).count()).fill_null(0.5).fill_nan(0.5)\n                         .cast(pl.Float32) for c in profile]).to_numpy()\n    reduced = PCA(16, random_state=seed).fit_transform(ranked - ranked.mean(0))\n    del ranked\n    out = {"request_id": req["request_id"]}\n    for k in [64, 512]:\n        out[f"pc{k}"] = MiniBatchKMeans(k, batch_size=8192, n_init=3, random_state=seed).fit_predict(reduced)\n    vectors = np.array([v if v is not None and len(v) == 32 else [0.0] * 32 for v in req[USER_VECTOR].to_list()],\n                       dtype=np.float32)\n    present = np.linalg.norm(vectors, axis=1) > 0\n    unit = vectors / np.maximum(np.linalg.norm(vectors, axis=1, keepdims=True), 1e-6)\n    labels = np.full(len(unit), -1)\n    labels[present] = MiniBatchKMeans(256, batch_size=8192, n_init=3, random_state=seed).fit_predict(unit[present])\n    out["uv256"] = labels\n    return pl.DataFrame(out).with_columns(pl.col("pc64", "pc512", "uv256").cast(pl.Int32)), len(profile)\n\n\ndef within_request_auc(requests, y, prediction):\n    """AUC of rows in mixed-outcome requests after removing each request\'s mean prediction."""\n    frame = pl.DataFrame({"r": requests, "y": y, "p": prediction}).with_columns(\n        (pl.col("p") - pl.col("p").mean().over("r")).alias("d"),\n        pl.col("y").sum().over("r").alias("s"), pl.len().over("r").alias("n"))\n    mixed = frame.filter((pl.col("s") > 0) & (pl.col("s") < pl.col("n")))\n    return float(roc_auc_score(mixed["y"], mixed["d"]))\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--data", type=Path, default=Path("data"))\n    parser.add_argument("--output", type=Path, default=Path("artifacts/local-t031"))\n    parser.add_argument("--variants", default="control,item_te,cf")\n    parser.add_argument("--learning-rate", type=float, default=0.05)\n    parser.add_argument("--rounds", type=int, default=4000)\n    parser.add_argument("--threads", type=int, default=8)\n    parser.add_argument("--rows", type=int, default=0, help="Keep 1/N of requests for smoke tests")\n    args = parser.parse_args()\n    args.output.mkdir(parents=True, exist_ok=True)\n    start = time.time()\n    assignments, n_profile = clusters(args.data, args.rows)\n    print(f"clusters ready ({n_profile} profile columns) {time.time() - start:.0f}s", flush=True)\n    schema = pl.scan_parquet(args.data / "train.parquet").collect_schema()\n    numeric = scalar_numeric(schema)\n    train = pl.scan_parquet(args.data / "train.parquet").with_row_index("row")\n    if args.rows:\n        train = train.filter(pl.col("request_id").hash(7) % args.rows == 0)\n    keys = sorted(set(ITEM_KEYS + LOW_CATEGORICAL), key=lambda c: int(c[7:]))\n    train = (train.select("row", "watched", "request_id", "feature1", "feature3", *keys,\n                          *[c for c in numeric if c not in keys]).collect()\n             .join(assignments, on="request_id", how="left"))\n    del assignments\n    rq, rq_names = request_features(train.select("row", "request_id", ITEM, *RANKED))\n    train = train.join(rq, on="row", how="left")\n    del rq\n    cross_names = []\n    for a, b in CROSSES:\n        name = f"x_{a}_{b}"\n        train = train.with_columns(pl.struct(a, b).hash(11).alias(name))\n        cross_names.append(name)\n    is_valid = (train["feature1"] >= CUTOFF).to_numpy()\n    fit_rows, valid_rows = train.filter(~pl.Series(is_valid)), train.filter(pl.Series(is_valid))\n    del train\n    gc.collect()\n    folds = list(GroupKFold(5).split(np.zeros(len(fit_rows)), groups=fit_rows["request_id"].to_numpy()))\n    te_fit, te_valid = {}, {}\n\n    def encode(columns):\n        names = []\n        for column in columns:\n            (a, an), (b, bn) = target_encode(fit_rows, valid_rows, column, folds)\n            te_fit[f"te_{column}"], te_valid[f"te_{column}"] = a, b\n            te_fit[f"ten_{column}"], te_valid[f"ten_{column}"] = np.log1p(an), np.log1p(bn)\n            names += [f"te_{column}", f"ten_{column}"]\n        return names\n    item_te = encode(ITEM_KEYS)\n    cf_te = encode(cross_names)\n    for column, (a, b) in zip(cross_names, CROSSES):\n        lift = f"lift_{column}"\n        te_fit[lift] = te_fit[f"te_{column}"] - te_fit[f"te_{b}"]\n        te_valid[lift] = te_valid[f"te_{column}"] - te_valid[f"te_{b}"]\n        cf_te.append(lift)\n    categories = {c: fit_rows[c].drop_nulls().unique().sort() for c in LOW_CATEGORICAL}\n    base = ["feature3", *[c for c in numeric if c not in categories], *LOW_CATEGORICAL, *rq_names]\n    groups = {"control": base, "item_te": base + item_te, "cf": base + item_te + cf_te}\n\n    def matrix(frame, te, columns):\n        parts = []\n        for c in columns:\n            if c in te:\n                parts.append(te[c].astype(np.float32))\n            elif c in categories:\n                parts.append(frame.select(pl.col(c).cast(pl.Enum(categories[c].to_list()), strict=False)\n                                          .to_physical().cast(pl.Float32))[c].to_numpy())\n            else:\n                parts.append(frame[c].cast(pl.Float32).to_numpy())\n        return np.ascontiguousarray(np.column_stack(parts), dtype=np.float32)\n\n    y_fit, y_valid = fit_rows["watched"].to_numpy(), valid_rows["watched"].to_numpy()\n    univariate = {c: float(roc_auc_score(y_valid, te_valid[c])) for c in item_te + cf_te if c.startswith(("te_", "lift_"))}\n    print("univariate holdout AUC:", json.dumps({k: round(v, 4) for k, v in univariate.items()}), flush=True)\n    results = {}\n    for name in args.variants.split(","):\n        columns = groups[name]\n        began = time.time()\n        x_fit, x_valid = matrix(fit_rows, te_fit, columns), matrix(valid_rows, te_valid, columns)\n        categorical = [i for i, c in enumerate(columns) if c in categories]\n        train_set = lgb.Dataset(x_fit, y_fit, feature_name=columns, categorical_feature=categorical, free_raw_data=True)\n        valid_set = lgb.Dataset(x_valid, y_valid, reference=train_set)\n        params = {\n            "objective": "binary", "metric": "auc", "learning_rate": args.learning_rate,\n            "num_leaves": 127, "min_data_in_leaf": 200, "feature_fraction": 0.5,\n            "bagging_fraction": 0.8, "bagging_freq": 1, "lambda_l2": 10,\n            "max_bin": 127, "seed": 42, "num_threads": args.threads, "verbosity": -1,\n            "cat_smooth": 20, "max_cat_to_onehot": 8,\n        }\n        model = lgb.train(params, train_set, args.rounds, valid_sets=[valid_set], valid_names=["holdout"],\n                          callbacks=[lgb.early_stopping(200, verbose=False), lgb.log_evaluation(250)])\n        prediction = model.predict(x_valid, num_iteration=model.best_iteration)\n        importance = sorted(zip(columns, model.feature_importance("gain")), key=lambda p: -p[1])[:30]\n        results[name] = {"holdout_auc": float(roc_auc_score(y_valid, prediction)),\n                         "within_request_auc": within_request_auc(valid_rows["request_id"], y_valid, prediction),\n                         "best_iteration": model.best_iteration, "features": len(columns),\n                         "seconds": time.time() - began, "top_gain": [[c, float(g)] for c, g in importance]}\n        np.save(args.output / f"valid-{name}.npy", prediction.astype(np.float32))\n        print(name, json.dumps({k: v for k, v in results[name].items() if k != "top_gain"}), flush=True)\n        print(" top:", [c for c, _ in importance[:15]], flush=True)\n        (args.output / "results.json").write_text(json.dumps({\n            "cutoff": CUTOFF.isoformat(), "valid_rows": int(len(y_valid)), "fit_rows": int(len(y_fit)),\n            "profile_columns": n_profile, "crosses": CROSSES, "univariate": univariate,\n            "early_stopping_on_holdout": True, "results": results,\n            "seconds": time.time() - start}, indent=2) + "\\n")\n        del x_fit, x_valid, train_set, valid_set, model\n        gc.collect()\n    np.save(args.output / "valid-row-ids.npy", valid_rows["row"].to_numpy())\n\n\nif __name__ == "__main__":\n    main()\n'}
frozen = Path("/kaggle/working/frozen-source")
frozen.mkdir()
for name, source in sources.items():
    (frozen / name).write_text(source)
(frozen / "t026-wide.json").write_text('{"wide_features": ["feature418", "feature419", "feature411", "feature410", "feature430", "feature431", "feature417", "feature352", "feature346", "feature480", "feature54", "feature493", "feature358", "feature515", "feature490", "feature429", "feature492", "feature510", "feature334", "feature511", "feature103", "feature39", "feature416", "feature484", "feature503", "feature150", "feature356", "feature151", "feature53", "feature337", "feature512", "feature478", "feature428", "feature514", "feature37", "feature513", "feature357", "feature376", "feature502", "feature439", "feature479", "feature507", "feature487", "feature333", "feature50", "feature320", "feature117", "feature126", "feature491", "feature56", "feature153"]}')
data = [p.parent for p in Path("/kaggle/input").rglob("train.parquet") if (p.parent / "sample_submission.csv").is_file()]
if len(data) != 1:
    raise RuntimeError(f"Expected competition data: {data}")
import xgboost
gpus = int(subprocess.run("nvidia-smi -L | wc -l", shell=True, capture_output=True, text=True).stdout.strip() or 1)
gpus = 1
print("xgboost", xgboost.__version__, "cpus", os.cpu_count(), "gpus", gpus, flush=True)
subprocess.run("nvidia-smi; free -m", shell=True, check=False)
monitor = subprocess.Popen("while true; do echo MEM $(date +%T) $(free -m | sed -n 2p); nvidia-smi --query-gpu=memory.used,utilization.gpu --format=csv,noheader | tr '\n' ' '; echo; sleep 120; done", shell=True)
matrices = "/tmp/t049-matrices"
try:
    subprocess.run([sys.executable, str(frozen / "xgb_tune.py"), "build", "--data", str(data[0]),
                    "--wide-from", str(frozen / "t026-wide.json"), "--matrices", matrices, "--cosines", "--entity"], check=True, cwd=frozen)
    configs = ['depth12_node', 'leaves511_eta02']
    workers = []
    for gpu in range(gpus):
        env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu)}
        log = open(f"/kaggle/working/worker{gpu}.log", "w")
        workers.append((subprocess.Popen([sys.executable, str(frozen / "xgb_tune.py"), "worker",
                                          "--matrices", matrices, "--output", "/kaggle/working/t049",
                                          "--configs", ",".join(configs[gpu::gpus])],
                                         cwd=frozen, env=env, stdout=log, stderr=subprocess.STDOUT), log))
    codes = [p.wait() for p, _ in workers]
    for gpu in range(gpus):
        print(f"===== worker{gpu} exit {codes[gpu]}", flush=True)
        print("\n".join(l for l in Path(f"/kaggle/working/worker{gpu}.log").read_text().splitlines()
                         if "WARNING" not in l), flush=True)
    if any(codes):
        raise RuntimeError(f"worker exit codes {codes}")
finally:
    monitor.kill()
